# Load data

In [29]:
import ast
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, f1_score, classification_report, confusion_matrix)

LABEL_COL = "final_label"
SEED = 42

df = pd.read_csv('Case4_Labeled.csv')
print(f"Loaded {len(df)} rows | columns: {list(df.columns)}")

Loaded 200 rows | columns: ['reviewer_id', 'store_location', 'latitude', 'longitude', 'date', 'month', 'year', 'title', 'review_text', 'star_rating', 'clean_text', 'topic_tokens', 'text_source', 'weak_label', 'final_label', 'manually_checked']


## Parse token 

In [30]:
def tokens_to_text(x):
    """topic_tokens is stored as a string like "['great', 'job']" -> 'great job'."""
    if isinstance(x, list):
        return " ".join(x)
    try:
        return " ".join(ast.literal_eval(str(x)))
    except (ValueError, SyntaxError):
        return ""

df["clean_text"] = df["clean_text"].fillna("").astype(str)
df["topic_text"] = df["topic_tokens"].apply(tokens_to_text)

# drop rows with no usable label / no text at all
df = df.dropna(subset=[LABEL_COL])
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)
df[LABEL_COL] = df[LABEL_COL].astype(int).values

print(f"\nLabel distribution ({LABEL_COL}):")
print(df[LABEL_COL].value_counts().sort_index().to_string(), "\n")


Label distribution (final_label):
final_label
0    149
1     51 



# Setup

In [32]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=[LABEL_COL]),
                                                    df[LABEL_COL].astype(int).values,
                                                    test_size=0.20, random_state=SEED,
                                                    stratify=df[LABEL_COL]
                                                )

# ------------------------------------------------------------------
# Feature sets (each = text column + vectorizer)
# ------------------------------------------------------------------
FEATURE_SETS = {
    "TFIDF unigrams (clean_text)": ("clean_text",
        lambda: TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.9, sublinear_tf=True)),
    "TFIDF uni+bigrams (clean_text)": ("clean_text",
        lambda: TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)),
    "TFIDF topic_tokens": ("topic_text",
        lambda: TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.9, sublinear_tf=True)),
}

# ------------------------------------------------------------------
# Hyperparameter grid
# k      : {1, 3, 5}
# metric : cosine vs euclidean
# algorithm='brute' is required for cosine on sparse matrices.
# ------------------------------------------------------------------
param_grid = {
    "knn__n_neighbors": [1, 3, 5],
    "knn__metric": ["cosine", "euclidean"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

summary = []
 
for name, (col, make_vec) in FEATURE_SETS.items():
    print("=" * 70)
    print(f"FEATURE SET: {name}")
    print("=" * 70)
 
    pipe = Pipeline([
        ("vec", make_vec()),
        ("knn", KNeighborsClassifier(algorithm="brute", n_jobs=-1)),
    ])
 
    gs = GridSearchCV(pipe, param_grid, cv=cv, scoring="f1_macro",
                      n_jobs=-1, refit=True, return_train_score=False)
    gs.fit(X_train[col], y_train)
 
    res = (pd.DataFrame(gs.cv_results_)
           [["param_knn__n_neighbors", "param_knn__metric",
             "mean_test_score", "std_test_score"]]
           .rename(columns={"param_knn__n_neighbors": "k",
                            "param_knn__metric": "metric",
                            "mean_test_score": "cv_f1_macro",
                            "std_test_score": "cv_std"})
           .sort_values("cv_f1_macro", ascending=False))
    print(res.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
 
    best = gs.best_params_
    y_pred = gs.predict(X_test[col])          # refit on full train with best params
    acc = accuracy_score(y_test, y_pred)
    f1m = f1_score(y_test, y_pred, average="macro")
 
    print(f"\nBEST CONFIG: k={best['knn__n_neighbors']}, metric={best['knn__metric']} "
          f"(CV macro-F1 = {gs.best_score_:.4f})")
    print(f"TEST accuracy = {acc:.4f} | TEST macro-F1 = {f1m:.4f}\n")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix (rows=true, cols=pred):")
    print(confusion_matrix(y_test, y_pred), "\n")
 
    summary.append({
        "feature_set": name,
        "best_k": best["knn__n_neighbors"],
        "best_metric": best["knn__metric"],
        "cv_f1_macro": round(gs.best_score_, 4),
        "test_accuracy": round(acc, 4),
        "test_f1_macro": round(f1m, 4),
    })

FEATURE SET: TFIDF unigrams (clean_text)
 k    metric  cv_f1_macro  cv_std
 1    cosine       0.6729  0.0850
 1 euclidean       0.6729  0.0850
 5    cosine       0.6622  0.0912
 5 euclidean       0.6622  0.0912
 3    cosine       0.6097  0.1003
 3 euclidean       0.6097  0.1003

BEST CONFIG: k=1, metric=cosine (CV macro-F1 = 0.6729)
TEST accuracy = 0.8250 | TEST macro-F1 = 0.7128

              precision    recall  f1-score   support

           0       0.83      0.97      0.89        30
           1       0.80      0.40      0.53        10

    accuracy                           0.82        40
   macro avg       0.81      0.68      0.71        40
weighted avg       0.82      0.82      0.80        40

Confusion matrix (rows=true, cols=pred):
[[29  1]
 [ 6  4]] 

FEATURE SET: TFIDF uni+bigrams (clean_text)
 k    metric  cv_f1_macro  cv_std
 5    cosine       0.6770  0.1357
 5 euclidean       0.6770  0.1357
 1    cosine       0.6585  0.0817
 1 euclidean       0.6585  0.0817
 3    cosine 

# Best config per feature set

In [33]:
summary_df = pd.DataFrame(summary).sort_values("cv_f1_macro", ascending=False)
print("=" * 70)
print("BEST CONFIG PER FEATURE SET")
print("=" * 70)
print(summary_df.to_string(index=False))
summary_df.to_csv("knn_best_config_per_feature_set.csv", index=False)

BEST CONFIG PER FEATURE SET
                   feature_set  best_k best_metric  cv_f1_macro  test_accuracy  test_f1_macro
            TFIDF topic_tokens       3      cosine       0.7573          0.775         0.6639
TFIDF uni+bigrams (clean_text)       5      cosine       0.6770          0.800         0.6537
   TFIDF unigrams (clean_text)       1      cosine       0.6729          0.825         0.7128


# Metric justification

WHY COSINE vs EUCLIDEAN?
- Features are sparse, high-dimensional TF-IDF vectors. Review length varies a lot
  ("Great job" vs. a paragraph), and we care about WHICH words appear, not how many.
  Cosine compares direction (word-usage profile) and ignores vector magnitude,
  so it is the natural choice for text.
- TfidfVectorizer L2-normalises every row, so ||a-b||^2 = 2 - 2*cos(a,b). On
  normalised vectors Euclidean and cosine therefore rank neighbours identically
  and should give (near-)identical results; differences only appear with
  distance-weighted voting or non-normalised features.
- Euclidean suffers more from the curse of dimensionality on sparse data, so
  cosine is the principled default. We still tune both and let CV confirm.